<a href="https://colab.research.google.com/github/nikitask14/adult-income-classification/blob/main/adult_income_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


*  Importing Pandas library
*  Importing the skicit library - sklearn and importing the dataset loader - fetch_openml




In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd


In [ ]:
!pip install ucimlrepo


In [ ]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=20)

X = adult.data.features
y = adult.data.targets

Loading the dataset and storing it into a bunch object called adult

In [ ]:
# adult = fetch_openml(name = "adult",version = 2, as_frame= True)

Diving the data into features and target label

In [ ]:
# X = adult.data

In [ ]:
# y = adult.target

Using info() method to get a comprehensive summary about the dataset.


1.  It tells us about the number of rows and columns.
2.  How many missing values are present
3.  What the numerical and categorical features are.
4.  The Non-Null tells us the number of actual values present for each of the feature columns.
5.  Columns with 48,842 non-null (like age or education) are completely full.



In [ ]:
type(X)
X.shape


(48842, 14)

In [ ]:
type(y)
y.shape


(48842, 1)

In [ ]:
y.columns

Index(['income'], dtype='object')

In [ ]:
y = y["income"]


### Dataset loading note

The original plan was to load the Adult dataset using `fetch_openml`, but OpenML repeatedly returned a 504 Gateway Timeout.

To avoid delaying the project, I loaded the same Adult/Census Income dataset from the UCI Machine Learning Repository using `ucimlrepo`.

After loading:
- `X` is a Pandas DataFrame with shape `(48842, 14)`
- `y` should be a 1D Pandas Series with shape `(48842,)

The UCI loader may initially return the target as a one-column DataFrame of shape `(48842, 1)`, so the target should be checked and, if necessary, converted to a Series before continuing.

In [ ]:
type(y)
y.shape

(48842,)

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
dtypes: int64(6), object(8)
memory usage: 5.2+ MB


Now, checking the number of null values in for each of the feature columns and sorting them by features with highest missing values to lowest.

In [ ]:
X.isnull().sum().sort_values(ascending = False)

,0
occupation,966
workclass,963
native-country,274
education,0
fnlwgt,0
age,0
marital-status,0
education-num,0
race,0
relationship,0


occupation has 2809 missing values, workclass	has 2799 missing values, native-country	has 857 missing values.

*   Occupation tells us whether person can earn 50,000 or not - Keep it
*   Working Class tells us about the sector - Which can be a possible leakage.
*   Native Country is a high cardinality feature- maybe we should not include it.





In [ ]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


 DATA AUDIT  
*   Check for class imbalance
*   Exclude high cardinality features as they would use too much memory when one hot encoded.
*   fnlwgt(doesn't seem to carry a lot of meaning), marital status not of much consequnce if earning or not, Race is controversial,
*   Check for data leakage - found none upfront.
*  Education and education-num seem similar - maybe keep education-num and drop out education to save dimensionality.








In [ ]:
X["native-country"].unique()

array(['United-States', 'Cuba', 'Jamaica', 'India', '?', 'Mexico',
       'South', 'Puerto-Rico', 'Honduras', 'England', 'Canada', 'Germany',
       'Iran', 'Philippines', 'Italy', 'Poland', 'Columbia', 'Cambodia',
       'Thailand', 'Ecuador', 'Laos', 'Taiwan', 'Haiti', 'Portugal',
       'Dominican-Republic', 'El-Salvador', 'France', 'Guatemala',
       'China', 'Japan', 'Yugoslavia', 'Peru',
       'Outlying-US(Guam-USVI-etc)', 'Scotland', 'Trinadad&Tobago',
       'Greece', 'Nicaragua', 'Vietnam', 'Hong', 'Ireland', 'Hungary',
       'Holand-Netherlands', nan], dtype=object)

In [ ]:
X["occupation"].unique()

array(['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners',
       'Prof-specialty', 'Other-service', 'Sales', 'Craft-repair',
       'Transport-moving', 'Farming-fishing', 'Machine-op-inspct',
       'Tech-support', '?', 'Protective-serv', 'Armed-Forces',
       'Priv-house-serv', nan], dtype=object)

In [ ]:
y.value_counts()

,count
income,
<=50K,24720
<=50K.,12435
>50K,7841
>50K.,3846


High class imbalance

---


Class 0 = 76%
Class 1 = 24%

We need to stratify the train_test split.

In [ ]:
retained_columns = ["occupation", "workclass", "native-country", "age",	"marital-status", "education-num", "race",
                    "relationship", "sex", "capital-gain", "capital-loss", "hours-per-week"]

In [ ]:
type(retained_columns)

list

In [ ]:
X_selected = X[retained_columns]
X.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
dtypes: int64(6), object(8)
memory usage: 5.2+ MB
